# 03 — Feature Engineering (VaxFlow)

สร้างฟีเจอร์อนุกรมเวลาต่อ (สาขา × ผลิตภัณฑ์) สำหรับโมเดลพยากรณ์ดีมานด์ 
รวมถึงค่า **Simple Moving Average** และ **Exponential Smoothing** ตามสมการใน Proposal §4.3

In [9]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
CLEAN = ROOT / 'data' / 'vaccine' / 'clean'
FEAT = ROOT / 'data' / 'vaccine' / 'features'; FEAT.mkdir(parents=True, exist_ok=True)
demand = pd.read_csv(CLEAN / 'demand_daily.csv', parse_dates=['date'])
demand = demand.sort_values(['hospital_id', 'product_id', 'date']).reset_index(drop=True)

## 1) ฟีเจอร์เวลา + lag + rolling (SMA)

`F(t+1) = (Σ D(t-i+1)) / n` — ค่าเฉลี่ยเคลื่อนที่ใช้เป็นพยากรณ์สภาวะปกติ

In [10]:
def add_features(g, sma_window=7):
    g = g.copy()
    g['dow'] = g['date'].dt.weekday
    g['is_weekend'] = (g['dow'] >= 5).astype(int)
    for lag in (1, 7, 14):
        g[f'lag_{lag}'] = g['demand'].shift(lag)
    g['sma_7'] = g['demand'].shift(1).rolling(7).mean()        # SMA (พยากรณ์ปกติ)
    g['sma_14'] = g['demand'].shift(1).rolling(14).mean()
    g['roll_std_7'] = g['demand'].shift(1).rolling(7).std()    # ความผันผวน
    return g

feat = demand.groupby(['hospital_id', 'product_id'], group_keys=False).apply(add_features)

C:\Users\jetan\AppData\Local\Temp\ipykernel_40928\4099612143.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  feat = demand.groupby(['hospital_id', 'product_id'], group_keys=False).apply(add_features)


## 2) Exponential Smoothing (สภาวะวิกฤต)

`F(t+1) = α·D(t) + (1-α)·F(t)` — ให้น้ำหนักข้อมูลล่าสุดมากกว่า เหมาะกับดีมานด์ที่ผันผวน

In [11]:
def exp_smoothing(series, alpha=0.4):
    f = np.zeros(len(series)); f[0] = series.iloc[0]
    for t in range(1, len(series)):
        f[t] = alpha * series.iloc[t - 1] + (1 - alpha) * f[t - 1]
    return f

feat['es_0.4'] = (feat.groupby(['hospital_id', 'product_id'])['demand']
                      .transform(lambda s: exp_smoothing(s, 0.4)))
feat = feat.dropna().reset_index(drop=True)
print('features:', feat.shape)
feat.head()

features: (1660, 13)


,date,hospital_id,product_id,demand,dow,is_weekend,lag_1,lag_7,lag_14,sma_7,sma_14,roll_std_7,es_0.4
0,2026-01-11,HOSP_001,VAX_MDV_01,8,6,1,8.0,9.0,9.0,13.714286,14.500000,4.347961,12.469157
1,2026-01-12,HOSP_001,VAX_MDV_01,18,0,0,8.0,18.0,18.0,13.571429,14.428571,4.540820,10.681494
2,2026-01-13,HOSP_001,VAX_MDV_01,18,1,0,18.0,18.0,16.0,13.571429,14.428571,4.540820,13.608896
3,2026-01-14,HOSP_001,VAX_MDV_01,17,2,0,18.0,12.0,20.0,13.571429,14.571429,4.540820,15.365338
4,2026-01-15,HOSP_001,VAX_MDV_01,18,3,0,17.0,13.0,19.0,14.285714,14.357143,4.644505,16.019203


## 3) บันทึกชุดฟีเจอร์

In [12]:
feat.to_csv(FEAT / 'demand_features.csv', index=False, encoding='utf-8-sig')
print('[saved]', (FEAT / 'demand_features.csv').resolve())
print('columns:', list(feat.columns))

[saved] C:\Work\LogisticsInnovationHackathon2026\VaxFlow\data\vaccine\features\demand_features.csv
columns: ['date', 'hospital_id', 'product_id', 'demand', 'dow', 'is_weekend', 'lag_1', 'lag_7', 'lag_14', 'sma_7', 'sma_14', 'roll_std_7', 'es_0.4']
